In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)

In [2]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "serious"
TOP_K_DRUGS = 500
TOP_K_REACTIONS = 300
TEST_SIZE = 0.2
RANDOM_STATE = 42

OUTCOME_REACTIONS = [
    "Death",
    "Sudden death",
    "Hospitalisation",
    "Completed suicide",
]

DRUG_SLOTS = [1, 2, 3]

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Krish\Desktop\BU\Codes\Projects\ADR\adr-prediction-system


In [3]:
dataset = pd.read_csv(PROCESSED_DATA_DIR / "faers_ml_dataset.csv", low_memory=False)
dataset["reactions"] = dataset["reactions"].fillna("")

print(f"Reports: {len(dataset):,}")
print(f"Serious: {dataset[TARGET].mean() * 100:.1f}%")
display(dataset[["primaryid", "age_years", "sex_clean", "n_drugs", "drug_1_ingredient", "reactions", TARGET]].head())

Reports: 400,513
Serious: 56.1%


,primaryid,age_years,sex_clean,n_drugs,drug_1_ingredient,reactions,serious
0,62150432,85.0,F,5,METHYLPREDNISOLONE SODIUM SUCCINATE,Acute pulmonary oedema|Cardiac failure|Hyperte...,1
1,62645954,59.0,M,4,OXYCODONE HYDROCHLORIDE,Retinal detachment,1
2,65308484,25.0,F,1,SULFASALAZINE,Intracranial pressure increased,1
3,65388687,30.0,F,17,ETANERCEPT,Arthralgia|Complication of pregnancy|Eclampsia...,1
4,65418874,NaN,F,12,ETANERCEPT,Exposure via breast milk|Gastrooesophageal ref...,1


In [4]:
remaining = dataset["reactions"].str.split("|").apply(lambda items: [r for r in items if r and r not in OUTCOME_REACTIONS])
only_outcome = remaining.str.len() == 0
print(f"Reports with only outcome-term reactions: {only_outcome.sum():,} ({only_outcome.mean() * 100:.1f}%), "
      f"of which serious: {dataset.loc[only_outcome, TARGET].mean() * 100:.1f}%")
dataset = dataset[~only_outcome].reset_index(drop=True)
print(f"Reports kept: {len(dataset):,}  (serious {dataset[TARGET].mean() * 100:.1f}%)")

Reports with only outcome-term reactions: 13,737 (3.4%), of which serious: 97.5%
Reports kept: 386,776  (serious 54.6%)


In [5]:
train_df, test_df = train_test_split(
    dataset,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=dataset[TARGET],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training reports: {len(train_df):,}  (serious {train_df[TARGET].mean() * 100:.1f}%)")
print(f"Test reports    : {len(test_df):,}  (serious {test_df[TARGET].mean() * 100:.1f}%)")

Training reports: 309,420  (serious 54.6%)
Test reports    : 77,356  (serious 54.6%)


In [6]:
slot_columns = [f"drug_{slot}_ingredient" for slot in DRUG_SLOTS]
drug_counts = train_df[slot_columns].stack().value_counts()
drug_vocab = drug_counts.head(TOP_K_DRUGS).index.tolist()

train_reactions = train_df["reactions"].str.split("|").explode()
train_reactions = train_reactions[train_reactions != ""]
reaction_counts = train_reactions.value_counts()
excluded = [r for r in OUTCOME_REACTIONS if r in reaction_counts.index]
reaction_counts = reaction_counts.drop(index=excluded)
reaction_vocab = reaction_counts.head(TOP_K_REACTIONS).index.tolist()

drug_coverage = drug_counts.head(TOP_K_DRUGS).sum() / drug_counts.sum() * 100
reaction_coverage = reaction_counts.head(TOP_K_REACTIONS).sum() / reaction_counts.sum() * 100

print(f"Ingredients: {len(drug_counts):,} distinct, top {TOP_K_DRUGS} cover {drug_coverage:.1f}% of mentions")
print(f"Reactions  : {len(reaction_counts):,} distinct, top {TOP_K_REACTIONS} cover {reaction_coverage:.1f}% of mentions")
print(f"Excluded outcome-like reactions: {excluded}")
print("\nMost common reactions kept:")
print(reaction_counts.head(10))

Ingredients: 8,221 distinct, top 500 cover 82.1% of mentions
Reactions  : 12,060 distinct, top 300 cover 64.3% of mentions
Excluded outcome-like reactions: ['Death', 'Sudden death', 'Hospitalisation', 'Completed suicide']

Most common reactions kept:
reactions
Off label use                  25843
Drug ineffective               18749
Fatigue                        15032
Product dose omission issue    13712
Nausea                         12201
Diarrhoea                      11959
Headache                        9910
Pain                            9385
Dyspnoea                        9332
Pruritus                        8889
Name: count, dtype: int64


In [7]:
drug_columns = ["ing_" + re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_") for name in drug_vocab]
reaction_columns = ["rx_" + re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_") for name in reaction_vocab]

all_columns = drug_columns + reaction_columns
assert len(set(all_columns)) == len(all_columns), "Two names cleaned to the same column name"

print(drug_columns[:3], "...")
print(reaction_columns[:3], "...")

['ing_DUPILUMAB', 'ing_TIRZEPATIDE', 'ing_ACETAMINOPHEN'] ...
['rx_Off_label_use', 'rx_Drug_ineffective', 'rx_Fatigue'] ...


In [8]:
drug_index = {name: i for i, name in enumerate(drug_vocab)}
reaction_index = {name: i for i, name in enumerate(reaction_vocab)}

feature_tables = {}
for split_name, df in [("train", train_df), ("test", test_df)]:
    n = len(df)
    rows = np.arange(n)
    features = pd.DataFrame(index=df.index)

    features["age_years"] = df["age_years"]
    features["age_missing"] = df["age_years"].isna().astype(np.uint8)
    for sex in ["F", "M", "Unknown"]:
        features[f"sex_{sex}"] = (df["sex_clean"] == sex).astype(np.uint8)

    features["n_drugs"] = df["n_drugs"].astype(int)
    drug_matrix = np.zeros((n, len(drug_vocab)), dtype=np.uint8)
    n_other_drugs = np.zeros(n, dtype=int)
    for slot in DRUG_SLOTS:
        names = df[f"drug_{slot}_ingredient"]
        position = names.map(drug_index)
        known = position.notna().to_numpy()
        drug_matrix[rows[known], position[known].astype(int)] = 1
        n_other_drugs += (names.notna() & position.isna()).to_numpy()
    features["n_other_drugs"] = n_other_drugs

    reaction_lists = df["reactions"].str.split("|")
    exploded = reaction_lists.explode()
    exploded = exploded[exploded.notna() & (exploded != "") & ~exploded.isin(OUTCOME_REACTIONS)]
    position = exploded.map(reaction_index)
    known = position.notna()
    reaction_matrix = np.zeros((n, len(reaction_vocab)), dtype=np.uint8)
    row_positions = df.index.get_indexer(exploded.index)
    reaction_matrix[row_positions[known.to_numpy()], position[known].astype(int).to_numpy()] = 1
    features["n_reactions"] = exploded.groupby(level=0).size().reindex(df.index, fill_value=0).astype(int)
    features["n_other_reactions"] = (~known).groupby(level=0).sum().reindex(df.index, fill_value=0).astype(int)

    drug_df = pd.DataFrame(drug_matrix, columns=drug_columns, index=df.index)
    reaction_df = pd.DataFrame(reaction_matrix, columns=reaction_columns, index=df.index)
    feature_tables[split_name] = pd.concat([features, drug_df, reaction_df], axis=1)

X_train = feature_tables["train"]
X_test = feature_tables["test"]
y_train = train_df[TARGET].astype(np.uint8)
y_test = test_df[TARGET].astype(np.uint8)
FEATURE_COLUMNS = X_train.columns.tolist()

print(f"X_train: {X_train.shape}  ({X_train.memory_usage(deep=True).sum() / 1024**2:.0f} MB)")
print(f"X_test : {X_test.shape}")

example = X_train.iloc[0]
print("\nExample report, non-zero features only:")
print(example[(example != 0) & example.notna()])
print(f"serious = {y_train.iloc[0]}")

X_train: (309420, 809)  (249 MB)
X_test : (77356, 809)

Example report, non-zero features only:
age_missing                      1.0
sex_M                            1.0
n_drugs                          2.0
n_other_drugs                    1.0
n_reactions                     22.0
n_other_reactions                8.0
ing_SODIUM_OXYBATE               1.0
rx_Drug_ineffective              1.0
rx_Fatigue                       1.0
rx_Pain                          1.0
rx_Dyspnoea                      1.0
rx_Decreased_appetite            1.0
rx_Blood_pressure_increased      1.0
rx_Somnolence                    1.0
rx_Gait_disturbance              1.0
rx_Feeling_abnormal              1.0
rx_Muscular_weakness             1.0
rx_Discomfort                    1.0
rx_Heart_rate_increased          1.0
rx_Intentional_dose_omission     1.0
rx_Vertigo                       1.0
Name: 0, dtype: float64
serious = 0


In [9]:
assert list(X_train.columns) == list(X_test.columns), "Train and test must have the same columns"
assert TARGET not in FEATURE_COLUMNS, "The target must not be a feature"
outcome_columns = ["rx_" + re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_") for name in OUTCOME_REACTIONS]
assert not set(outcome_columns) & set(reaction_columns), "Outcome-like reactions must be excluded"
assert X_train[drug_columns + reaction_columns].isin([0, 1]).all().all(), "Yes/no columns must be 0 or 1"
drugs_in_slots = train_df[slot_columns].notna().sum(axis=1)
assert (drugs_in_slots == X_train[drug_columns].sum(axis=1) + X_train["n_other_drugs"]).all(), "Every drug counted once"
assert (X_train["n_reactions"] == X_train[reaction_columns].sum(axis=1) + X_train["n_other_reactions"]).all(), "Every reaction counted once"
assert not set(train_df["primaryid"]) & set(test_df["primaryid"]), "No report in both sets"
assert (X_train["n_reactions"] > 0).all() and (X_test["n_reactions"] > 0).all(), "Every report must keep at least one real reaction"

print("All sanity checks passed.")

All sanity checks passed.


In [10]:
train_out = pd.concat([train_df[["primaryid"]], X_train, y_train.rename(TARGET)], axis=1)
test_out = pd.concat([test_df[["primaryid"]], X_test, y_test.rename(TARGET)], axis=1)
train_out.to_parquet(PROCESSED_DATA_DIR / "serious_train.parquet", index=False)
test_out.to_parquet(PROCESSED_DATA_DIR / "serious_test.parquet", index=False)

config = {
    "target": TARGET,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "feature_columns": FEATURE_COLUMNS,
    "ingredients": drug_vocab,
    "ingredient_columns": drug_columns,
    "ingredient_report_counts": [int(drug_counts[d]) for d in drug_vocab],
    "reactions": reaction_vocab,
    "reaction_columns": reaction_columns,
    "reaction_report_counts": [int(reaction_counts[r]) for r in reaction_vocab],
    "excluded_reactions": OUTCOME_REACTIONS,
}
(MODELS_DIR / "serious_feature_config.json").write_text(json.dumps(config, indent=2))

for name in ["data/processed/serious_train.parquet", "data/processed/serious_test.parquet", "models/serious_feature_config.json"]:
    path = PROJECT_ROOT / name
    print(f"Saved {name}  ({path.stat().st_size / 1024**2:.1f} MB)")
print(f"\nFeatures: {len(FEATURE_COLUMNS)} = 7 patient/drug counts + 2 reaction counts + {len(drug_columns)} ingredients + {len(reaction_columns)} reactions")

Saved data/processed/serious_train.parquet  (6.2 MB)
Saved data/processed/serious_test.parquet  (2.1 MB)
Saved models/serious_feature_config.json  (0.1 MB)

Features: 809 = 7 patient/drug counts + 2 reaction counts + 500 ingredients + 300 reactions
